In [1]:
# Task 1: Write functions in day12.ipynb, then move them into reader.py, validator.py and transformer.py
from pathlib import Path

Path("reader.py").write_text('''"""Read a CSV file."""
import csv


def read_csv(path: str) -> list[dict]:
    """Return the rows of a CSV file as a list of dictionaries."""
    rows = []
    with open(path, newline="", encoding="utf-8-sig") as file:
        for row in csv.DictReader(file):
            rows.append(dict(row))
    return rows
''')

Path("validator.py").write_text('''"""Check that the needed columns are there."""


def validate_required_columns(rows: list[dict], required: list) -> list:
    """Return the names of the columns that are missing."""
    missing = []
    for name in required:
        if rows and name not in rows[0]:
            missing.append(name)
    return missing
''')

Path("transformer.py").write_text('''"""Rename columns and change their values."""
from datetime import datetime

COLUMN_MAPPING = {
    "order_id": "order_id",
    "freight_value": "freight",
    "price": "price",
    "shipping_limit_date": "shipping_limit_date",
}

TIMESTAMP_COLUMNS = ["shipping_limit_date", "order_purchase_timestamp"]


def to_snake_case(name: str) -> str:
    """Change a column name to snake_case."""
    return name.strip().lower().replace(" ", "_").replace("-", "_").replace(".", "_")


def map_columns(rows: list[dict], mapping: dict) -> list[dict]:
    """Rename the columns using a dictionary."""
    new_rows = []
    for row in rows:
        new_row = {}
        for key, value in row.items():
            new_row[mapping.get(key, key)] = value
        new_rows.append(new_row)
    return new_rows


def convert_types(rows: list[dict]) -> list[dict]:
    """Make price and freight floats and the dates datetimes."""
    for row in rows:
        row["price"] = float(row["price"])
        row["freight"] = float(row["freight"])
        for name in TIMESTAMP_COLUMNS:
            if row.get(name):
                row[name] = datetime.strptime(row[name], "%Y-%m-%d %H:%M:%S")
    return rows


def calculate_order_totals(rows: list[dict]) -> dict:
    """Add price and freight for every order id."""
    totals = {}
    for row in rows:
        order_id = row["order_id"]
        if order_id not in totals:
            totals[order_id] = 0
        totals[order_id] += float(row["price"]) + float(row["freight"])
    return totals
''')

print("Made reader.py, validator.py and transformer.py")

Made reader.py, validator.py and transformer.py


In [2]:
# Task 2: Import your modules into the notebook (use %autoreload)
%load_ext autoreload
%autoreload 2

import reader
import transformer
import validator

print("Imported the three modules")

Imported the three modules


In [3]:
# Task 3: Write read_csv(path) returning a list of dictionaries
items = reader.read_csv("../archive/olist_order_items_dataset.csv")
print("Rows:", len(items))
print("First row:", items[0])

Rows: 112650
First row: {'order_id': '00010242fe8c5a6d1ba2dd792cb16214', 'order_item_id': '1', 'product_id': '4244733e06e7ecb4970a6e2683c13e61', 'seller_id': '48436dade18ac8b2bce089ec2a041202', 'shipping_limit_date': '2017-09-19 09:45:35', 'price': '58.90', 'freight_value': '13.29'}


In [4]:
# Task 4: Write a function that converts column names to snake_case
for name in ["Freight Value", "order-id", "Product.Name"]:
    print(name, "->", transformer.to_snake_case(name))

Freight Value -> freight_value
order-id -> order_id
Product.Name -> product_name


In [5]:
# Task 5: Create a column-mapping dictionary to a standard schema
print("Mapping:", transformer.COLUMN_MAPPING)
standard_rows = transformer.map_columns(items, transformer.COLUMN_MAPPING)
print("Mapped row:", standard_rows[0])

Mapping: {'order_id': 'order_id', 'freight_value': 'freight', 'price': 'price', 'shipping_limit_date': 'shipping_limit_date'}
Mapped row: {'order_id': '00010242fe8c5a6d1ba2dd792cb16214', 'order_item_id': '1', 'product_id': '4244733e06e7ecb4970a6e2683c13e61', 'seller_id': '48436dade18ac8b2bce089ec2a041202', 'shipping_limit_date': '2017-09-19 09:45:35', 'price': '58.90', 'freight': '13.29'}


In [6]:
# Task 6: Convert price and freight to float, and timestamps to datetime
typed_rows = transformer.convert_types(standard_rows)
print(type(typed_rows[0]["price"]), typed_rows[0]["price"])
print(type(typed_rows[0]["freight"]), typed_rows[0]["freight"])
print(type(typed_rows[0]["shipping_limit_date"]), typed_rows[0]["shipping_limit_date"])

<class 'float'> 58.9
<class 'float'> 13.29
<class 'datetime.datetime'> 2017-09-19 09:45:35


In [7]:
# Task 7: Calculate order totals per order_id using a dictionary
totals = transformer.calculate_order_totals(typed_rows)
print("Orders:", len(totals))
print("First three:", list(totals.items())[:3])

Orders: 98666
First three: [('00010242fe8c5a6d1ba2dd792cb16214', 72.19), ('00018f77f2f0320c557190d7a144bdd3', 259.83), ('000229ec398224ef6ca0657da4fc703e', 216.87)]


In [8]:
# Task 8: Write pytest tests for three functions and run them from the notebook with !pytest
from pathlib import Path

Path("test_pipeline.py").write_text(r'''from reader import read_csv
from transformer import calculate_order_totals, to_snake_case


def test_read_csv(tmp_path):
    file = tmp_path / "x.csv"
    file.write_text("a,b\n1,2\n")
    assert read_csv(str(file)) == [{"a": "1", "b": "2"}]


def test_to_snake_case():
    assert to_snake_case("Freight Value") == "freight_value"


def test_calculate_order_totals():
    rows = [
        {"order_id": "A", "price": 10, "freight": 2},
        {"order_id": "A", "price": 5, "freight": 1},
    ]
    assert calculate_order_totals(rows) == {"A": 18}
''')

!pytest -q

...                                                                      [100%]
3 passed in 0.07s


In [9]:
# Task 9: Add type hints and docstrings; open a PR with passing tests
print(reader.read_csv.__annotations__)
print(reader.read_csv.__doc__)
print("Tests passed, ready for the PR")

{'path': <class 'str'>, 'return': list[dict]}
Return the rows of a CSV file as a list of dictionaries.
Tests passed, ready for the PR


In [10]:
# Task 10: Load the column mapping from a JSON config file instead of code
import json
from pathlib import Path

Path("column_mapping.json").write_text(json.dumps(transformer.COLUMN_MAPPING, indent=2))
config_mapping = json.loads(Path("column_mapping.json").read_text())
config_rows = transformer.map_columns(items, config_mapping)
config_totals = transformer.calculate_order_totals(config_rows)
print("Config:", config_mapping)
print("Same totals:", config_totals == totals)

Config: {'order_id': 'order_id', 'freight_value': 'freight', 'price': 'price', 'shipping_limit_date': 'shipping_limit_date'}
Same totals: True
